# Modeling V1 — Colab: hai baseline TF-IDF

Notebook này huấn luyện **hai pipeline độc lập trên cùng Gold V1**:

1. TF-IDF + **Logistic Regression** (25-class multinomial, có predict_proba/softmax).
2. TF-IDF + **LinearSVC** (25-class, decision scores, **không** có xác suất mặc định).

Có tùy chọn thêm Logistic Regression `class_weight="balanced"` làm thí nghiệm thứ ba.

**Không tạo lại Gold, không fit TF-IDF trên validation/test, không tự đánh giá test.**
Chạy tuần tự để tránh sử dụng RAM song song trên Colab CPU. Kết quả và model được ghi **một lần** vào Drive; các artifact đã hoàn chỉnh/khớp cấu hình sẽ được xác minh và bỏ qua khi chạy lại.

Chạy các cell từ trên xuống. Sử dụng Google Drive có thư mục `vietnamese-multilabel-topic-classification` đang chứa `04_gold/manifests`.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/Ko4rut/vietnamese-multimodal-moderation.git"
BRANCH = "feat/colab-modeling-baselines-v1"
PROJECT_DIR = Path("/content/vmm_modeling_project")

if (PROJECT_DIR / ".git").is_dir():
    dirty = subprocess.run(
        ["git", "-C", str(PROJECT_DIR), "status", "--porcelain"],
        capture_output=True, text=True, check=True,
    )
    if dirty.stdout.strip():
        raise RuntimeError("Repository Colab có thay đổi chưa commit; hãy kiểm tra trước khi checkout.")
    subprocess.run(["git", "-C", str(PROJECT_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(
        ["git", "-C", str(PROJECT_DIR), "checkout", "-B", BRANCH, "FETCH_HEAD"],
        check=True,
    )
else:
    if PROJECT_DIR.exists() and any(PROJECT_DIR.iterdir()):
        raise RuntimeError(f"{PROJECT_DIR} đã tồn tại nhưng không phải Git repository.")
    subprocess.run(
        ["git", "clone", "--single-branch", "--branch", BRANCH, REPO_URL, str(PROJECT_DIR)],
        check=True,
    )

revision = subprocess.check_output(
    ["git", "-C", str(PROJECT_DIR), "rev-parse", "--short", "HEAD"], text=True
).strip()
print(f"Repository: {BRANCH} @ {revision}")


In [ ]:
import subprocess
import sys

# Chỉ cài dependency phục vụ modeling; không cần tải lại dataset.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", ".[modeling]"],
    cwd=str(PROJECT_DIR), check=True,
)

SRC_DIR = (PROJECT_DIR / "src").resolve()
if not (SRC_DIR / "vietnamese_topic_classification" / "__init__.py").is_file():
    raise FileNotFoundError(f"Không tìm thấy Python package: {SRC_DIR}")
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import sklearn
from vietnamese_topic_classification.config import get_domain_labels
from vietnamese_topic_classification.data.gold import GoldReader

print("Python:", sys.version.split()[0])
print("scikit-learn:", sklearn.__version__)
print("Số domain labels:", len(get_domain_labels()))
assert len(get_domain_labels()) == 25


In [ ]:
from google.colab import drive

MOUNT = "/content/vtsnlp_drive"
drive.mount(MOUNT)

# Đúng tên Data Lake đã dùng trong Colab Gold runner; chỉnh tại đây nếu đã đổi thư mục.
DATA_LAKE_ROOT = Path(MOUNT) / "MyDrive" / "vietnamese-multilabel-topic-classification"
MANIFEST_DIR = DATA_LAKE_ROOT / "04_gold" / "manifests"
OUTPUT_ROOT = DATA_LAKE_ROOT / "experiments" / "tfidf_baselines_v1"

if not MANIFEST_DIR.is_dir():
    raise FileNotFoundError(
        f"Không tìm thấy {MANIFEST_DIR}. Hãy kiểm tra tài khoản Drive hoặc DATA_LAKE_ROOT."
    )

# Nếu chỉ có một snapshot Gold, notebook tự chọn; nếu nhiều, hãy chỉ định filename.
GOLD_MANIFEST_NAME = None
available_manifests = sorted(p.name for p in MANIFEST_DIR.glob("gold_manifest_v1_*.json"))
print("Gold manifests:", available_manifests)
if GOLD_MANIFEST_NAME is None:
    if len(available_manifests) != 1:
        raise ValueError(
            "Cần đúng một Gold manifest, hoặc gán GOLD_MANIFEST_NAME bằng tên file "
            f"trong danh sách ở trên. Hiện có {len(available_manifests)}."
        )
    GOLD_MANIFEST_NAME = available_manifests[0]
if Path(GOLD_MANIFEST_NAME).name != GOLD_MANIFEST_NAME:
    raise ValueError("GOLD_MANIFEST_NAME phải là tên file JSON, không phải đường dẫn.")

reader = GoldReader(DATA_LAKE_ROOT, GOLD_MANIFEST_NAME)
verified = reader.verify()  # Checksum và số dòng của cả ba split, không fit/đọc nội dung test.
if reader.manifest["label_to_id"] != {
    label: idx for idx, label in enumerate(get_domain_labels())
}:
    raise AssertionError("Gold label mapping không khớp 25 nhãn canonical.")
if verified["split_rows"]["train"] < 2 or verified["split_rows"]["validation"] < 1:
    raise ValueError("Gold train/validation quá nhỏ để modeling.")

print("GOLD VERIFIED:", verified)
print("Manifest:", reader.manifest_path)
print("Experiments destination:", OUTPUT_ROOT)


In [ ]:
import hashlib
import json
import os

SEED = 42
MAX_FEATURES = 100_000
MIN_DF = 2
RUN_BALANCED = False  # Đổi True nếu muốn train thêm LR class_weight='balanced'.

# Hai baseline mặc định: LogisticRegression và LinearSVC, cùng TF-IDF.
EXPERIMENT_SPECS = [
    {"slug": "tfidf_logreg", "classifier": "logistic_regression", "class_weight": "none"},
    {"slug": "tfidf_linear_svc", "classifier": "linear_svc", "class_weight": "none"},
]
if RUN_BALANCED:
    EXPERIMENT_SPECS.append(
        {"slug": "tfidf_logreg_balanced",
         "classifier": "logistic_regression", "class_weight": "balanced"}
    )

if MAX_FEATURES < 1 or MIN_DF < 1:
    raise ValueError("MAX_FEATURES và MIN_DF phải dương.")

manifest_sha256 = hashlib.sha256(reader.manifest_path.read_bytes()).hexdigest()
for spec in EXPERIMENT_SPECS:
    spec["name"] = f"{spec['slug']}_v1_{manifest_sha256[:10]}"
    spec["dir"] = OUTPUT_ROOT / spec["name"]

def verify_existing_experiment(spec):
    """Chỉ skip khi model + reports hoàn chỉnh, cùng Gold source và hyperparams."""
    directory = spec["dir"]
    expected_files = ("model.joblib", "config.json", "metadata.json", "validation_metrics.json")
    missing = [name for name in expected_files if not (directory / name).is_file()]
    if missing:
        raise FileExistsError(f"Experiment tồn tại nhưng thiếu {missing}: {directory}")
    config = json.loads((directory / "config.json").read_text(encoding="utf-8"))
    metadata = json.loads((directory / "metadata.json").read_text(encoding="utf-8"))
    report = json.loads((directory / "validation_metrics.json").read_text(encoding="utf-8"))
    expected_weight = None if spec["class_weight"] == "none" else "balanced"
    if (
        config.get("gold_manifest_name") != GOLD_MANIFEST_NAME
        or config.get("seed") != SEED
        or config.get("classifier", {}).get("kind") != spec["classifier"]
        or config.get("classifier", {}).get("params", {}).get("class_weight") != expected_weight
        or config.get("features", {}).get("kind") != "tfidf"
        or config.get("features", {}).get("params", {}).get("max_features") != MAX_FEATURES
        or config.get("features", {}).get("params", {}).get("min_df") != MIN_DF
        or config.get("features", {}).get("params", {}).get("ngram_range") != [1, 2]
        or metadata.get("lineage", {}).get("gold_manifest_sha256") != manifest_sha256
        or not isinstance(report.get("macro_f1"), (int, float))
    ):
        raise ValueError(f"Artifact không khớp cấu hình/Gold hiện tại: {directory}")
    return report

# Preflight toàn bộ trước khi bắt đầu để không tự ghi đè artifacts cũ.
existing_reports = {}
for spec in EXPERIMENT_SPECS:
    if spec["dir"].exists():
        existing_reports[spec["name"]] = verify_existing_experiment(spec)
        print("VERIFIED EXISTING — SKIP:", spec["dir"])
    else:
        print("READY TO TRAIN:", spec["name"])
print("Tổng số baseline:", len(EXPERIMENT_SPECS))


In [ ]:
# Gọi training CLI gốc, tránh duplicate sklearn fit/metric logic trong notebook.
# Chạy TUẦN TỰ để hạn chế RAM trên Colab CPU.
env = os.environ.copy()
env["PYTHONPATH"] = os.pathsep.join(
    part for part in [str(SRC_DIR), env.get("PYTHONPATH", "")] if part
)

for spec in EXPERIMENT_SPECS:
    if spec["name"] in existing_reports:
        print("SKIPPED (verified):", spec["name"])
        continue
    command = [
        sys.executable, str(PROJECT_DIR / "scripts/06_train_baseline.py"),
        "--manifest-name", GOLD_MANIFEST_NAME,
        "--data-lake-root", str(DATA_LAKE_ROOT),
        "--output-root", str(OUTPUT_ROOT),
        "--name", spec["name"],
        "--classifier", spec["classifier"],
        "--class-weight", spec["class_weight"],
        "--max-features", str(MAX_FEATURES),
        "--min-df", str(MIN_DF),
        "--seed", str(SEED),
    ]
    print("TRAINING:", spec["name"], "(", spec["classifier"], ")")
    subprocess.run(command, cwd=str(PROJECT_DIR), env=env, check=True)
    existing_reports[spec["name"]] = verify_existing_experiment(spec)
    print("SAVED:", spec["dir"])
print("ALL BASELINE EXPERIMENTS READY")


In [ ]:
# Validation ONLY — chưa sử dụng test để chọn mô hình.
from IPython.display import Markdown, display

rows = []
for spec in EXPERIMENT_SPECS:
    metrics = verify_existing_experiment(spec)
    if len(metrics.get("per_class_f1", {})) != 25:
        raise AssertionError(f"Thiếu per-class F1 (25 nhãn): {spec['name']}")
    if len(metrics.get("confusion_matrix", [])) != 25:
        raise AssertionError(f"Confusion matrix không đủ 25 hàng: {spec['name']}")
    rows.append({
        "name": spec["name"],
        "classifier": spec["classifier"],
        "macro_f1": metrics["macro_f1"],
        "weighted_f1": metrics["weighted_f1"],
        "accuracy": metrics["accuracy"],
        "artifact_dir": str(spec["dir"]),
    })

rows.sort(key=lambda row: (-row["macro_f1"], row["name"]))
headers = "| Rank | Model | Macro-F1 | Weighted-F1 | Accuracy |"
divider = "|---:|---|---:|---:|---:|"
lines = [
    f"| {idx} | {row['name']} | {row['macro_f1']:.4f} "
    f"| {row['weighted_f1']:.4f} | {row['accuracy']:.4f} |"
    for idx, row in enumerate(rows, 1)
]
display(Markdown("\n".join([headers, divider, *lines])))
print("Best on VALIDATION Macro-F1:", rows[0]["name"])
print("Selected artifact:", rows[0]["artifact_dir"])
print("Test set CHƯA được đánh giá.")


In [ ]:
# Demo xác suất Softmax chỉ cho Logistic Regression.
# Chỉ load artifact do chính notebook này tạo/đã kiểm chứng; joblib là định dạng pickle.
import joblib

logreg_spec = next(
    spec for spec in EXPERIMENT_SPECS if spec["classifier"] == "logistic_regression"
)
pipeline = joblib.load(logreg_spec["dir"] / "model.joblib")
example_text = "Đội tuyển bóng đá Việt Nam giành chiến thắng trong trận đấu."
probabilities = pipeline.predict_proba([example_text])[0]
labels = get_domain_labels()
ranked = sorted(
    ((labels[int(label_id)], float(probabilities[idx]))
     for idx, label_id in enumerate(pipeline.classes_)),
    key=lambda item: item[1], reverse=True,
)
print("Ví dụ:", example_text)
print("Top 5 dự đoán (model Logistic Regression):")
for domain, probability in ranked[:5]:
    print(f"{domain:30s}: {probability:.3%}")
print("Tổng xác suất:", sum(probabilities))
# LinearSVC không có predict_proba mặc định: decision scores KHÔNG là xác suất.


## Sau khi chạy xong

- Mỗi thí nghiệm được lưu ở `DATA_LAKE_ROOT/experiments/tfidf_baselines_v1/<experiment_name>/`: `model.joblib`, `config.json`, `metadata.json`, `validation_metrics.json`.
- Cột **Macro-F1** trên validation được dùng để chọn baseline; so sánh per-class F1 để tìm lớp yếu.
- **Chưa chạy `scripts/07_evaluate_model.py`**: chỉ đánh giá Gold test một lần sau khi đã chốt mô hình và không tiếp tục tuning dựa trên test.
- Nếu muốn chạy thêm LR weighted, đổi `RUN_BALANCED=True` ở cell cấu hình và chạy lại cấu hình/training/comparison; notebook sẽ bỏ qua hai experiment cũ đã xác minh.
- Nếu muốn thay hyperparameters (`MAX_FEATURES`, `MIN_DF`, v.v.), đổi tên experiment hoặc phiên bản; notebook sẽ từ chối ghi đè model đã có.

**Lưu ý:** 25 nhãn VTSNLP đến từ nhãn nguồn tự động, chưa phải ground truth được kiểm duyệt thủ công; cần audit dữ liệu và lỗi dự đoán trước khi kết luận khoa học.